# Data Quality Tests for Wheelie Data Warehouse

This notebook contains data quality tests for all dimension tables and bridge tables.
Each test validates uniqueness constraints and referential integrity.

In [ ]:
# ==============================================================================
# TEST CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================
import logging
from pyspark.sql.functions import col

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("data_quality_tests")

def get_table(table_name: str):
    """Helper to load table from data warehouse."""
    return spark.table(f"wheelie.data_warehouse.{table_name}")

def assert_uniqueness(table_name: str, column_name: str):
    """Assert that a column has all unique values."""
    df = get_table(table_name)
    total_count = df.count()
    distinct_count = df.select(column_name).distinct().count()
    duplicates_count = total_count - distinct_count

    if duplicates_count > 0:
        duplicates = df.groupBy(column_name).count().filter(col("count") > 1)
        logger.error(f"Duplicate values found in {table_name}.{column_name}:")
        duplicates.show(10)

    assert total_count == distinct_count, \
        f"{table_name}.{column_name}: Found {duplicates_count} duplicate(s). Expected all {total_count} values to be unique."

def assert_referential_integrity(source_table: str, source_col: str, target_table: str, target_col: str):
    """Assert that all values in source column exist in target column (excludes nulls)."""
    source_df = get_table(source_table)
    target_df = get_table(target_table)

    # Filter out nulls from source before checking
    missing = source_df.filter(col(source_col).isNotNull()) \
        .select(col(source_col).alias("key")) \
        .distinct() \
        .join(
            target_df.select(col(target_col).alias("key")),
            "key",
            "left_anti"
        )

    missing_count = missing.count()
    total_count = source_df.filter(col(source_col).isNotNull()).select(source_col).distinct().count()

    if missing_count > 0:
        logger.error(f"Missing references from {source_table}.{source_col} to {target_table}.{target_col}:")
        missing.show(10)

    assert missing_count == 0, \
        f"Referential integrity violation: {missing_count} out of {total_count} keys in {source_table}.{source_col} do not exist in {target_table}.{target_col}"

logger.info("=" * 70)
logger.info("DATA QUALITY TEST FRAMEWORK LOADED")
logger.info("=" * 70)


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: DIMENSIONS
# ==============================================================================

def test_dim_date_key_unique():
    """Test that date_key is unique in dim_date."""
    assert_uniqueness("dim_date", "date_key")

def test_dim_service_date_key_unique():
    """Test that service_date_key is unique in dim_service_date."""
    assert_uniqueness("dim_service_date", "service_date_key")

def test_dim_staff_staff_key_unique():
    """Test that staff_key is unique in dim_staff."""
    assert_uniqueness("dim_staff", "staff_key")

def test_dim_store_store_key_unique():
    """Test that store_key is unique in dim_store."""
    assert_uniqueness("dim_store", "store_key")

def test_dim_store_store_id_unique():
    """Test that store_id is unique in dim_store."""
    assert_uniqueness("dim_store", "store_id")

def test_dim_car_car_key_unique():
    """Test that car_key is unique in dim_car."""
    assert_uniqueness("dim_car", "car_key")

def test_dim_customer_customer_key_unique():
    """Test that customer_key is unique in dim_customer."""
    assert_uniqueness("dim_customer", "customer_key")

def test_dim_customer_customer_id_unique():
    """Test that customer_id is unique in dim_customer."""
    assert_uniqueness("dim_customer", "customer_id")

def test_dim_equipment_equipment_key_unique():
    """Test that equipment_key is unique in dim_equipment."""
    assert_uniqueness("dim_equipment", "equipment_key")

logger.info("✅ Dimension test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: BRIDGE TABLES
# ==============================================================================

def test_bridge_car_equipment_car_key_unique():
    """Test that car_key is unique in bridge_car_equipment."""
    assert_uniqueness("bridge_car_equipment", "car_key")

def test_bridge_equipment_group_equipment_group_key_exists():
    """Test that all equipment_group_key values exist in bridge_car_equipment."""
    assert_referential_integrity(
        source_table="bridge_equipment_group_equipment",
        source_col="equipment_group_key",
        target_table="bridge_car_equipment",
        target_col="equipment_group_key"
    )

def test_bridge_equipment_group_equipment_key_exists():
    """Test that all equipment_key values exist in dim_equipment."""
    assert_referential_integrity(
        source_table="bridge_equipment_group_equipment",
        source_col="equipment_key",
        target_table="dim_equipment",
        target_col="equipment_key"
    )

logger.info("✅ Bridge table test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: FACT TABLES
# ==============================================================================

def test_fact_service_key_unique():
    """Test that service_key is unique in fact_service."""
    assert_uniqueness("fact_service", "service_key")

def test_fact_service_car_key_exists():
    """Test that all car_key values in fact_service exist in dim_car."""
    assert_referential_integrity("fact_service", "car_key", "dim_car", "car_key")

def test_fact_service_date_key_exists():
    """Test that all service_date_key values in fact_service exist in dim_service_date."""
    assert_referential_integrity("fact_service", "service_date_key", "dim_service_date", "service_date_key")

def test_fact_rental_key_unique():
    """Test that rental_key is unique in fact_rental."""
    assert_uniqueness("fact_rental", "rental_key")

def test_fact_rental_customer_key_exists():
    """Test that all customer_key values in fact_rental exist in dim_customer."""
    assert_referential_integrity("fact_rental", "customer_key", "dim_customer", "customer_key")

def test_fact_rental_car_key_exists():
    """Test that all car_key values in fact_rental exist in dim_car."""
    assert_referential_integrity("fact_rental", "car_key", "dim_car", "car_key")

def test_fact_rental_staff_key_exists():
    """Test that all staff_key values in fact_rental exist in dim_staff."""
    assert_referential_integrity("fact_rental", "staff_key", "dim_staff", "staff_key")

def test_fact_rental_store_key_exists():
    """Test that all store_key values in fact_rental exist in dim_store."""
    assert_referential_integrity("fact_rental", "store_key", "dim_store", "store_key")

def test_fact_rental_date_keys_exist():
    """Test that all date keys in fact_rental exist in dim_date (excluding nulls)."""
    # Test rental_date_key
    assert_referential_integrity("fact_rental", "rental_date_key", "dim_date", "date_key")

    # Test return_date_key (nulls filtered by assert_referential_integrity)
    assert_referential_integrity("fact_rental", "return_date_key", "dim_date", "date_key")

    # Test payment_date_key (nulls filtered by assert_referential_integrity)
    assert_referential_integrity("fact_rental", "payment_date_key", "dim_date", "date_key")

    # Test payment_deadline_date_key
    assert_referential_integrity("fact_rental", "payment_deadline_date_key", "dim_date", "date_key")

logger.info("✅ Fact table test definitions loaded")


In [ ]:
# ==============================================================================
# RUN ALL TESTS
# ==============================================================================

logger.info("\n" + "=" * 70)
logger.info("EXECUTING ALL DATA QUALITY TESTS")
logger.info("=" * 70 + "\n")

# Collect all test functions
test_functions = [
    # Dimension tests
    ("dim_date", test_dim_date_key_unique),
    ("dim_service_date", test_dim_service_date_key_unique),
    ("dim_staff", test_dim_staff_staff_key_unique),
    ("dim_store (store_key)", test_dim_store_store_key_unique),
    ("dim_store (store_id)", test_dim_store_store_id_unique),
    ("dim_car", test_dim_car_car_key_unique),
    ("dim_customer (customer_key)", test_dim_customer_customer_key_unique),
    ("dim_customer (customer_id)", test_dim_customer_customer_id_unique),
    ("dim_equipment", test_dim_equipment_equipment_key_unique),

    # Bridge table tests
    ("bridge_car_equipment", test_bridge_car_equipment_car_key_unique),
    ("bridge_equipment_group (FK to car_equipment)", test_bridge_equipment_group_equipment_group_key_exists),
    ("bridge_equipment_group (FK to equipment)", test_bridge_equipment_group_equipment_key_exists),

    # Fact table tests
    ("fact_service (service_key)", test_fact_service_key_unique),
    ("fact_service (FK to car)", test_fact_service_car_key_exists),
    ("fact_service (FK to service_date)", test_fact_service_date_key_exists),
    ("fact_rental (rental_key)", test_fact_rental_key_unique),
    ("fact_rental (FK to customer)", test_fact_rental_customer_key_exists),
    ("fact_rental (FK to car)", test_fact_rental_car_key_exists),
    ("fact_rental (FK to staff)", test_fact_rental_staff_key_exists),
    ("fact_rental (FK to store)", test_fact_rental_store_key_exists),
    ("fact_rental (FK to date)", test_fact_rental_date_keys_exist),
]

passed = 0
failed = 0
failed_tests = []

for test_name, test_func in test_functions:
    try:
        logger.info(f"Running: {test_name} - {test_func.__doc__}")
        test_func()
        passed += 1
        logger.info(f"✅ PASS: {test_name}\n")
    except AssertionError as e:
        failed += 1
        failed_tests.append({
            "test": test_name,
            "description": test_func.__doc__,
            "error": str(e)
        })
        logger.error(f"❌ FAIL: {test_name}")
        logger.error(f"   {str(e)}\n")
    except Exception as e:
        failed += 1
        failed_tests.append({
            "test": test_name,
            "description": test_func.__doc__,
            "error": f"Unexpected error: {str(e)}"
        })
        logger.error(f"❌ ERROR: {test_name}")
        logger.error(f"   Unexpected error: {str(e)}\n")

total = len(test_functions)

logger.info("=" * 70)
logger.info("TEST EXECUTION SUMMARY")
logger.info("=" * 70)
logger.info(f"Total Tests: {total}")
logger.info(f"Passed: {passed} ✅")
logger.info(f"Failed: {failed} ❌")
logger.info(f"Success Rate: {(passed/total*100):.1f}%")

if failed > 0:
    logger.error("\n" + "=" * 70)
    logger.error("FAILED TESTS DETAILS")
    logger.error("=" * 70)
    for test in failed_tests:
        logger.error(f"\n❌ {test['test']}")
        logger.error(f"   Description: {test['description']}")
        logger.error(f"   Error: {test['error']}")
    logger.error("\n" + "=" * 70)
    logger.error(f"⚠️  {failed} TEST(S) FAILED - REVIEW REQUIRED")
    logger.error("=" * 70)

    # Exit with failure status for pipeline orchestration
    dbutils.notebook.exit(f"FAILURE: {failed}/{total} tests failed")
else:
    logger.info("\n" + "=" * 70)
    logger.info("🎉 ALL DATA QUALITY TESTS PASSED!")
    logger.info("=" * 70)
    logger.info("Coverage:")
    logger.info("  • Dimensions: 7 tables (date, service_date, staff, store, car, customer, equipment)")
    logger.info("  • Bridge tables: 3 tables (staff_hierarchy, car_equipment, equipment_group)")
    logger.info("  • Fact tables: 2 tables (service, rental)")
    logger.info("  • Total assertions: 21 tests")
    logger.info("=" * 70)

    # Exit with success status for pipeline orchestration
    dbutils.notebook.exit(f"SUCCESS: All {total} tests passed")
